<a href="https://colab.research.google.com/github/kenzoyanome/brazilian_ecommerce/blob/main/bra_ecommerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olist
Identificar oportunidades de mejora en el negocio

### Bases de datos
customers_dataset:
  - customer_id
  - customer_unique_id
  - customer_zip_code_prefix
  - customer_city
  - customer_state

geolocation_dataset (**divided in 3 files**):
  - geolocation_zip_code_prefix
  - geolocation_lat
  - geolocation_lng
  - geolocation_city
  - geolocation_state

order_items_dataset:
  - order_id
  - order_item_id
  - product_id
  - seller_id
  - shipping_limit_date
  - price
  - freight_value

order_payments_dataset:
  - order_id
  - payment_sequential
  - payment_type
  - payment_installments
  - payment_value

order_reviews_dataset:  
  - review_id
  - order_id
  - review_score
  - review_comment_title
  - review_comment_message
  - review_creation_date
  - review_answer_timestamp

orders_dataset:
  - order_id
  - customer_id
  - order_status
  - order_purchase_timestamp
  - order_approved_at
  - order_delivered_carrier_date
  - order_delivered_customer_date
  - order_estimated_delivery_date

product_category_name_translation:
  - product_category_name
  - product_category_name_english

products_dataset:
  - product_id
  - product_category_name
  - product_name_lenght
  - product_description_lenght
  - product_photos_qty
  - product_weight_g
  - product_length_cm
  - product_height_cm
  - product_width_cm

sellers_dataset:
  - seller_id
  - seller_zip_code_prefix
  - seller_city
  - seller_state

### Preguntas de negocio
Ventas  
¿Cuánto vende Olist?  
¿Cómo evolucionan las ventas mensualmente?  
¿Cuál es el ticket promedio?  
¿Qué estados generan más ingresos?  

Clientes  
¿Quiénes son los clientes más valiosos?  
¿Cuántas compras realiza cada cliente?  
¿Hay señales de abandono?  

Productos  
¿Qué categorías generan más ingresos?  
¿Cuáles tienen mejores reseñas?  
¿Existe relación entre precio y satisfacción?  

Logística  
¿Cuánto tarda Olist en entregar?  
¿Qué porcentaje de pedidos llega tarde?  
¿La entrega tardía afecta la calificación?  

Vendedores  
¿Qué vendedores generan más ingresos?  
¿Cuáles tienen mejores calificaciones?  
¿Qué vendedores tienen problemas de entrega?  

---
Fase 1 - Python: limpieza, auditoría de calidad de datos, feature engineering (tiempos de entrega, retrasos, valor de pedido, etc.) y EDA con seaborn/matplotlib.

Fase 2 - SQL (PostgreSQL): funnel de conversión, cohortes, análisis de retención y rentabilidad usando window functions.

Fase 3 - Power BI: modelo de datos, medidas DAX, y diseño de 2-3 dashboards (Ejecutivo, Logística, Clientes).

---

### SQL
JOIN  
GROUP BY  
CASE WHEN  
CTE  
funciones de fecha  
subconsultas  
WINDOW FUNCTIONS  
COALESCE  
NULLIF  
agregaciones  

### Python
limpieza  
EDA  
detección de outliers  
análisis de distribución  
correlaciones  
análisis estadístico  
visualizaciones  
si encontramos una pregunta interesante, podemos hacer incluso una prueba estadística para comprobar si un hallazgo es realmente significativo.

### Power BI
Finalmente construiremos un dashboard con varias páginas:  
1. Executive Overview  
Ventas | órdenes | clientes | ticket promedio | entregas  
2. Sales & Products  
Categorías | productos | estados | evolución temporal  
3. Customer Analytics  
Clientes | RFM | frecuencia | valor monetario  
4. Logistics  
Tiempo de entrega | retrasos | estados | vendedores  
5. Customer Satisfaction  
Reviews | rating | retrasos vs satisfacción  

### Conclusiones
Deberiamos de termionar con un proyecto en GitHub, algo como:  
"Analicé aproximadamente 100,000 órdenes de un marketplace brasileño, integrando información de clientes, productos, vendedores, pagos, logística y satisfacción. Utilicé SQL para transformar y relacionar las fuentes, Python para el análisis exploratorio y Power BI para desarrollar un dashboard ejecutivo. Identifiqué los principales factores asociados con ventas, retrasos logísticos y satisfacción del cliente, y generé recomendaciones accionables."

In [ ]:
# IMPORTAR LIBRERÍAS RELEVANTES
import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt
import seaborn           as sns

from scipy.stats                  import pointbiserialr, chi2_contingency, ttest_ind, levene, pointbiserialr, chi2_contingency, ttest_ind, ttest_1samp, mannwhitneyu, shapiro
from statsmodels.stats.proportion import proportions_ztest
# OTRAS CONFIGURACIONES: EN GENERAL, EN BLOQUE DE IMPORTS, SUELE EXISTIR UN ESPACIO DONDE SE HACEN DEFINICIONES GENERALES
#                       COMO ELIMINAR CIERTOS "WARNINGS" MOLESTOS, QUITAR LÍMITES DE PANDAS SOBRE CUANTAS FILAS O COLUMNAS
#                       MUESTRA AL MOMENTO DE IMPIRMIR EL DATAFRAME, DEFINIR ESTILOS PRE-DEFINIDOS PARA GRÁFICOS, ETC.
import warnings                                                  # MANEJO DE WARNINGS - ADVERTENCIAS
warnings.filterwarnings('ignore', category=FutureWarning)        # IGNORAR WARNINGS MOLESTOS
pd.set_option('display.max_columns', None)                       # ELIMINA LIMITES DE PANDAS PARA MOSTRAR COLUMNAS
pd.set_option('display.max_rows', None)                          # ELIMINA LIMITES DE PANDAS PARA MOSTRAR FILAS
pd.set_option('display.max_colwidth', None)                      # AUTOAJUSTA ANCHO DE COLUMNAS
pd.set_option('display.float_format', lambda x: '%.3f' % x)      # PERMITE EVADIR EL MOSTRAR NÚMEROS CON NOTACIÓN CINETÍFICA
plt.rcParams['figure.dpi'] = 140                                 # NIVEL DE RESOLUCIÓN.

In [ ]:
# IMPORTAR ARCHIVOS
# to upload files from collab
# You can upload files from your local machine to Colab using
# google.colab.files.upload()
# When you run this code, it will prompt you to select the file(s) from your local file system.

#from google.colab import files
#uploaded = files.upload()
#for fn in uploaded.keys():
#  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

# IMPORTAR ARCHIVOS
customers           = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/customers_dataset.csv')
geolocation_a       = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/geolocation_dataset_a.csv')
geolocation_b       = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/geolocation_dataset_b.csv')
geolocation_c       = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/geolocation_dataset_c.csv')
order_items         = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/order_items_dataset.csv')
order_payments      = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/order_payments_dataset.csv')
order_reviews       = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/order_reviews_dataset.csv')
orders              = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/orders_dataset.csv')
pc_name_translation = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/product_category_name_translation.csv')
products            = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/products_dataset.csv')
sellers             = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/sellers_dataset.csv')

HTTPError: HTTP Error 404: Not Found